# Airline Data Checks and Cleaning

This notebook checks and cleans the 2025 US airline dataset.

The file is large, so it is processed in smaller chunks.

In [ ]:
import pandas as pd
import zipfile
from pathlib import Path

RAW_FILE = Path("flights_2025_raw.zip")
CLEANED_CSV = Path("flights_2025_cleaned.csv")
CLEANED_ZIP = Path("flights_2025_cleaned.zip")
CHUNK_SIZE = 250_000

## 1. Preview the data

In [ ]:
sample = pd.read_csv(RAW_FILE, nrows=5)
sample

In [ ]:
print("Columns:", sample.shape[1])
sample.dtypes

## 2. Check the complete dataset

In [ ]:
total_rows = 0
missing_values = pd.Series(0, index=sample.columns, dtype="int64")
airlines = set()

for chunk in pd.read_csv(RAW_FILE, chunksize=CHUNK_SIZE):
    total_rows += len(chunk)
    missing_values += chunk.isna().sum()
    airlines.update(chunk["Reporting_Airline"].dropna().unique())

print("Rows:", f"{total_rows:,}")
print("Airlines:", sorted(airlines))

In [ ]:
missing_percent = (missing_values / total_rows * 100).round(2)
missing_percent[missing_percent > 0].sort_values(ascending=False)

Missing flight times are normal for cancelled or diverted flights.  
Missing delay causes mean that no cause was reported, so they will be replaced with 0.

## 3. Clean the data

In [ ]:
airline_names = {
    "AA": "American Airlines",
    "AS": "Alaska Airlines",
    "B6": "JetBlue Airways",
    "DL": "Delta Air Lines",
    "F9": "Frontier Airlines",
    "G4": "Allegiant Air",
    "HA": "Hawaiian Airlines",
    "MQ": "Envoy Air",
    "NK": "Spirit Airlines",
    "OH": "PSA Airlines",
    "OO": "SkyWest Airlines",
    "UA": "United Airlines",
    "WN": "Southwest Airlines",
    "YX": "Republic Airways",
}

cancellation_reasons = {
    "A": "Carrier",
    "B": "Weather",
    "C": "National Air System",
    "D": "Security",
}

In [ ]:
def clean_chunk(df):
    # Use simple column names
    df.columns = (
        df.columns.str.strip()
        .str.replace(r"(?<!^)(?=[A-Z])", "_", regex=True)
        .str.lower()
        .str.replace("__", "_", regex=False)
    )

    # Remove duplicate rows
    df = df.drop_duplicates()

    # Clean text columns
    text_columns = df.select_dtypes(include="object").columns
    for column in text_columns:
        df[column] = df[column].str.strip()

    # Convert the date
    df["flight_date"] = pd.to_datetime(df["flight_date"], errors="coerce")

    # Convert flags to whole numbers
    flag_columns = ["cancelled", "diverted", "dep_del15", "arr_del15"]
    for column in flag_columns:
        df[column] = df[column].fillna(0).astype("int8")

    # Replace missing delay causes with 0
    delay_causes = [
        "carrier_delay", "weather_delay", "n_a_s_delay",
        "security_delay", "late_aircraft_delay"
    ]
    df[delay_causes] = df[delay_causes].fillna(0)

    # Add clear labels
    df["airline_name"] = df["reporting_airline"].map(airline_names)
    df["cancellation_reason"] = (
        df["cancellation_code"].map(cancellation_reasons).fillna("Not Cancelled")
    )
    df["route"] = df["origin"] + " - " + df["dest"]

    # Add the scheduled departure hour
    df["departure_hour"] = (df["c_r_s_dep_time"] // 100).astype("int16")
    df.loc[df["departure_hour"] == 24, "departure_hour"] = 0

    # Add a time-of-day group
    df["time_of_day"] = "Night"
    df.loc[df["departure_hour"].between(5, 11), "time_of_day"] = "Morning"
    df.loc[df["departure_hour"].between(12, 16), "time_of_day"] = "Afternoon"
    df.loc[df["departure_hour"].between(17, 21), "time_of_day"] = "Evening"

    # Add one final flight status
    df["flight_status"] = "On Time"
    df.loc[df["arr_delay"] >= 15, "flight_status"] = "Delayed"
    df.loc[df["diverted"] == 1, "flight_status"] = "Diverted"
    df.loc[df["cancelled"] == 1, "flight_status"] = "Cancelled"

    df["on_time"] = (df["flight_status"] == "On Time").astype("int8")
    df["delayed"] = (df["flight_status"] == "Delayed").astype("int8")

    return df

In [ ]:
# Clean and save every chunk
CLEANED_CSV.unlink(missing_ok=True)
first_chunk = True
cleaned_rows = 0

for chunk in pd.read_csv(RAW_FILE, chunksize=CHUNK_SIZE):
    cleaned_chunk = clean_chunk(chunk)
    cleaned_chunk.to_csv(
        CLEANED_CSV,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False,
    )
    cleaned_rows += len(cleaned_chunk)
    first_chunk = False

print("Cleaned rows:", f"{cleaned_rows:,}")

## 4. Validate the cleaned data

In [ ]:
cleaned_sample = pd.read_csv(CLEANED_CSV, nrows=5)
cleaned_sample

In [ ]:
status_counts = pd.Series(dtype="int64")
final_rows = 0

for chunk in pd.read_csv(CLEANED_CSV, usecols=["flight_status"], chunksize=CHUNK_SIZE):
    final_rows += len(chunk)
    status_counts = status_counts.add(chunk["flight_status"].value_counts(), fill_value=0)

print("Final rows:", f"{final_rows:,}")
print(status_counts.astype("int64"))

## 5. Compress the cleaned CSV

In [ ]:
CLEANED_ZIP.unlink(missing_ok=True)

with zipfile.ZipFile(CLEANED_ZIP, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as zip_file:
    zip_file.write(CLEANED_CSV, arcname=CLEANED_CSV.name)

print("Created:", CLEANED_ZIP)
print("ZIP size:", round(CLEANED_ZIP.stat().st_size / 1024**2, 1), "MB")

## Result

The cleaned dataset is ready for exploratory data analysis and Power BI.